In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Intermediate ESI 2, 3, 4 Random Forest Regressor (`models/rf_extreme.ipynb`)

This notebook trains a **Random Forest Model** to classify intermediate acuity levels **ESI 2, ESI 3, and ESI 4**:
- **Dataset Exclusion**: **ESI 1 and ESI 5 rows are completely removed** prior to partitioning into Train, Validation, and Test splits.
- **Target Output**: Predicts whether a patient is **ESI 2**, **ESI 3**, or **ESI 4** (`"2"`, `"3"`, `"4"`).
- **Complete Case Filtering**: Removes any row with at least one NULL/NA feature across all splits.
- **Model Engine**: **Random Forest** (`randomForest::randomForest`).
- **Evaluation Metrics**: **Accuracy**, **Multi-Class ROC-AUC**, and **Log Loss**.
- **Model Export**: Saved to `deploy/rf_extreme_model.rds`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(caret)
library(randomForest)
library(dplyr)
library(ggplot2)
library(pROC)

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}

config <- fromJSON(config_path)

cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data, Exclude ESI 1 & 5 Rows, & Apply Complete Case Analysis
# ---------------------------------------------------------
set.seed(config$training$random_state)

data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}

cat("Loading dataset from:", data_file, "...\n")

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))

raw_df <- get(data_obj_name, envir = data_env)

base_features <- config$features$base_features
if (is.null(base_features)) {
  base_features <- config$features$data_name
}
target_col <- config$classes$target_col

df <- raw_df[, c(intersect(base_features, names(raw_df)), target_col)]

if ("gender" %in% names(df)) {
  df$gender <- ifelse(as.character(df$gender) == "Male", 1, 0)
}

# Strict Complete Case Analysis: Remove rows with ANY NULL/NA features
df <- na.omit(df)

# FILTER OUT ESI 1 AND ESI 5 ROWS COMPLETELY
raw_esi <- as.character(df[[target_col]])
esi234_idx <- which(raw_esi %in% c("2", "3", "4"))
df <- df[esi234_idx, ]

# Create Target Factor for ESI 2, 3, 4
df$target_esi234 <- factor(as.character(df[[target_col]]), levels = c("2", "3", "4"))

cat(sprintf("Intermediate ESI 2,3,4 Random Forest Dataset Ready (ESI 1 & 5 Excluded): %d rows x %d cols\n",
            nrow(df), ncol(df)))
cat("Target Distribution:\n")
print(table(df$target_esi234))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Data Partitioning & Feature Scaling
# ---------------------------------------------------------
set.seed(config$training$random_state)

test_size <- config$training$test_size
val_size  <- config$training$val_size

# Stratified Test split (15%)
in_train_val <- createDataPartition(df$target_esi234, p = 1 - test_size, list = FALSE)
train_val_df <- df[in_train_val, ]
test_df      <- df[-in_train_val, ]

# Stratified Validation split (15%)
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_esi234, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]

# Standardize numeric features (center and scale)
numeric_cols <- names(train_df)[sapply(train_df, is.numeric)]
preproc <- preProcess(train_df[, numeric_cols], method = c("center", "scale"))

train_df[, numeric_cols] <- predict(preproc, train_df[, numeric_cols])
val_df[, numeric_cols]   <- predict(preproc, val_df[, numeric_cols])
test_df[, numeric_cols]  <- predict(preproc, test_df[, numeric_cols])

cat(sprintf("Complete Case ESI 2,3,4 Partition sizes:\n  Train: %d rows\n  Val:   %d rows\n  Test:  %d rows\n",
            nrow(train_df), nrow(val_df), nrow(test_df)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Train ESI 2, 3, 4 Random Forest Model
# ---------------------------------------------------------
set.seed(config$training$random_state)

feat_names <- setdiff(names(train_df), c(target_col, "target_esi234"))
formula_rf <- as.formula(paste("target_esi234 ~", paste(feat_names, collapse = " + ")))

cat("Training Random Forest Model for ESI 2, 3, 4...\n")
rf_model <- randomForest(formula_rf, data = train_df, ntree = 300, importance = TRUE)

cat("Random Forest ESI 2, 3, 4 training complete!\n")
print(rf_model)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Evaluate Scoring Metrics (Accuracy, Multi-Class ROC-AUC, Log Loss)
# ---------------------------------------------------------
calc_log_loss <- function(actual_factor, prob_matrix, eps = 1e-15) {
  prob_matrix <- pmax(pmin(prob_matrix, 1 - eps), eps)
  prob_matrix <- prob_matrix / rowSums(prob_matrix)
  classes <- colnames(prob_matrix)
  N <- length(actual_factor)
  
  log_probs <- numeric(N)
  for (i in 1:N) {
    act_cls <- as.character(actual_factor[i])
    if (act_cls %in% classes) {
      log_probs[i] <- log(prob_matrix[i, act_cls])
    } else {
      log_probs[i] <- log(eps)
    }
  }
  return(-mean(log_probs))
}
evaluate_esi234_rf <- function(model, data, set_name) {
  prob_matrix <- predict(model, newdata = data, type = "prob")
  target_classes <- levels(data$target_esi234)
  
  max_idx <- max.col(prob_matrix, ties.method = "first")
  pred_factor <- factor(colnames(prob_matrix)[max_idx], levels = target_classes)
  actual_factor <- factor(data$target_esi234, levels = target_classes)
  
  cm <- confusionMatrix(pred_factor, actual_factor)
  acc <- as.numeric(cm$overall["Accuracy"])
  
  roc_auc <- tryCatch({
    as.numeric(pROC::multiclass.roc(actual_factor, prob_matrix)$auc)
  }, error = function(e) NA)
  
  log_loss <- calc_log_loss(actual_factor, prob_matrix)
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   RANDOM FOREST ESI 2, 3, 4 - %s SET BENCHMARK\n", toupper(set_name)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Accuracy   : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  ROC-AUC    : %.4f\n", roc_auc))
  cat(sprintf("  Log Loss   : %.4f\n", log_loss))
  cat("\nConfusion Matrix:\n")
  print(cm$table)
  cat(sprintf("============================================================\n\n"))
}
# Evaluate on Validation set
evaluate_esi234_rf(rf_model, val_df, "Validation")
# Evaluate on Test set
evaluate_esi234_rf(rf_model, test_df, "Test")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Save Extreme Random Forest Model Artifacts
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)

model_path <- file.path(deploy_dir, "rf_extreme_model.rds")
saveRDS(list(model = rf_model, preproc = preproc), file = model_path)
cat("Random Forest ESI 2, 3, 4 model saved to:", model_path, "\n")